# SR-EV Simulation Pipeline Demo

Demonstrates `hmt_v3.srev.pipeline` end-to-end: takes an already-generated SR-EV `config-N.txt`, splits monomers into two compartments (heterochromatin / euchromatin, via `srev.compartments`), and runs each compartment as its own independent **channel** through primary/secondary antibody labeling, Markov blinking photophysics, PSF + camera-noise rendering, and ground-truth bookkeeping -- producing two TIFF stacks (one per channel) plus two saved ground-truth run directories, ready for ThunderSTORM.

This mirrors `hmt_v1`'s original two-dye setup (one dye per histone mark/compartment) but goes through the corrected `hmt_v3.srev` pipeline instead of the legacy MATLAB code.

**Scope note**: `pipeline.run_pipeline` renders exactly one channel per call, by design -- see `hmt_v3/srev/pipeline.py`'s module docstring. Two such calls (first half of this notebook) give two *separate* movies that never interact. The second half uses `pipeline.run_multichannel_pipeline` instead, which is what `hmt_v1`'s `master2.m` actually simulated: both dyes blinking in the **same** frames of one zeroth-order (spatial) stack plus one first-order (spectral) stack, separable afterwards only by their emission spectra -- i.e. with RainbowSTORM.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy.spatial import cKDTree

from hmt_v3 import srev

## Load a config and classify compartments on the FULL monomer cloud

Compartment classification (coordination number, `srev.compartments`) counts real neighbors within a fixed radius, so it needs to run on the FULL, uncropped monomer cloud -- doing it after cropping to a small patch would undercount monomers near the crop boundary and bias them toward euchromatin. Crop for speed only AFTER classifying (next cell).

In [ ]:
CONFIG_PATH = Path("hmt_v1/Simulation/config-1.txt")  # point this at any config-N.txt(.gz)

# Prefer resolve_srev_run_params (copy-parameters.txt / folder name) when available; it returns
# whatever it can find without raising just because some fields are missing (e.g. config-1.txt's
# parent folder isn't an "SRRW-*" name and has no sibling copy-parameters.txt), so fall back to
# length_unit_to_nm=10 (Umin) -- correct for Cangnano et al. 2024 SR-EV runs -- via .get() rather
# than a try/except (a real ValueError here, from disagreeing sources, should still surface).
params = srev.io.resolve_srev_run_params(CONFIG_PATH)
length_unit_to_nm = params.get("length_unit_to_nm", 10.0)
if "length_unit_to_nm" not in params:
    print(f"note: could not resolve length_unit_to_nm from '{CONFIG_PATH.parent.name}' "
          f"(no matching SRRW-* folder name or copy-parameters.txt) -- defaulting to 10.0 nm")

monomers_full, meta = srev.io.parse_config_txt(CONFIG_PATH, length_unit_to_nm=length_unit_to_nm)
monomers_full = srev.compartments.classify_compartments(monomers_full)

print(f"{len(monomers_full)} monomers parsed from {CONFIG_PATH}")
print(monomers_full["compartment"].value_counts().to_dict())

## Crop to a fast-to-render patch

SR-EV configs can have 10^5-10^6 monomers -- far more than fit in a typical camera FOV or are worth simulating for a demo run. This takes the `N_PATCH` monomers nearest the cloud's centroid (robust to whatever physical scale this particular config turns out to have) and keeps the `compartment` column computed above intact.

In [ ]:
N_PATCH = 6000

centroid = monomers_full[["x [nm]", "y [nm]", "z [nm]"]].to_numpy().mean(axis=0)
tree = cKDTree(monomers_full[["x [nm]", "y [nm]", "z [nm]"]].to_numpy())
_, idx = tree.query(centroid, k=min(N_PATCH, len(monomers_full)))
patch_df = monomers_full.iloc[idx].reset_index(drop=True).copy()

print(f"patch: {len(patch_df)} monomers")
print(patch_df["compartment"].value_counts().to_dict())

## Configure the two channels

Each `SrEvSimConfig` nests every stage's own config (`LabelingConfig`, `PhotophysicsConfig`, `CameraConfig`, `RenderConfig`) so a parameter is declared once and used everywhere it's read -- no separately-drifting copies. The two channels below differ in labeling density and dye brightness (standing in for two different antibody/dye pairs, e.g. an anti-H3K27me3/Alexa647 heterochromatin channel vs. an anti-H3K27ac/CF680 euchromatin channel) and use different RNG seeds so they don't share randomness.

In [ ]:
config_heterochromatin = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=500, mean_photons_per_frame=300.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="heterochromatin",
    seed=1,
)

config_euchromatin = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=500, mean_photons_per_frame=450.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="euchromatin",
    seed=2,
)

## Run both channels

Each call to `run_pipeline` is fully self-contained: it filters to the requested compartment, runs labeling -> photophysics, auto-sizes the camera frame to fit the labeled emitters (`render.fit_frame_to_points`), renders + noises the movie, writes a BigTIFF stack, and saves the ground-truth tables (Parquet + `run_config.json`) -- checking referential integrity before writing anything.

In [ ]:
OUTPUT_ROOT = Path("sr_ev_pipeline_demo_output")

result_het = srev.pipeline.run_pipeline(
    patch_df, config_heterochromatin, OUTPUT_ROOT / "heterochromatin",
    source_path=CONFIG_PATH, input_meta=meta,
)
result_eu = srev.pipeline.run_pipeline(
    patch_df, config_euchromatin, OUTPUT_ROOT / "euchromatin",
    source_path=CONFIG_PATH, input_meta=meta,
)

for name, result in (("heterochromatin", result_het), ("euchromatin", result_eu)):
    print(f"--- {name} ---")
    print(f"  monomers used:      {len(result['monomers'])}")
    print(f"  primaries:          {len(result['primaries'])}")
    print(f"  secondaries:        {len(result['secondaries'])}")
    print(f"  fluorophores:       {len(result['fluorophores'])}")
    print(f"  blink events:       {len(result['blink_events'])}")
    print(f"  frame_shape (px):   {result['frame_shape']}")
    print(f"  TIFF stack:         {result['written']['tiff']}")

## Visualize both channels

A max-intensity projection across the movie for each channel, side by side -- a quick sanity check that emitters actually landed inside the frame and produced signal above the camera baseline, before handing the stacks to ThunderSTORM.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (name, result) in zip(axes, (("heterochromatin", result_het), ("euchromatin", result_eu))):
    stack = tifffile.imread(result["written"]["tiff"])
    mip = stack.max(axis=0)
    im = ax.imshow(mip, cmap="inferno")
    ax.set_title(f"{name}\n{stack.shape[0]} frames, {stack.shape[1]}x{stack.shape[2]} px")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ADU (max over stack)")
fig.tight_layout()
plt.show()

## Inspect the ground truth

`resolved_localizations_gt` is the direct comparison target for a real ThunderSTORM run on these stacks: one row per blink event, with the emitter's TRUE static position joined in (not what any fitter would recover).

In [ ]:
loaded_het = srev.ground_truth.load_run(OUTPUT_ROOT / "heterochromatin")
loaded_eu = srev.ground_truth.load_run(OUTPUT_ROOT / "euchromatin")

print("heterochromatin resolved_localizations_gt:")
display(loaded_het["resolved_localizations_gt"].head())

print("euchromatin resolved_localizations_gt:")
display(loaded_eu["resolved_localizations_gt"].head())

## Dual-color sSMLM: both dyes in ONE spatial + spectral stack pair

`hmt_v1/Simulation/master2.m` did not produce one movie per dye. It rendered every AF647- and CF680-labeled emitter together into a single zeroth-order (spatial) stack (`blinking_sim_unrealistic.m`) and a single first-order (spectral) stack in which each emitter's PSF is shifted along x by the grating's dispersion and convolved with its dye's emission spectrum (`make_first_order.m` + `make_real_fist.m`); the two dyes were only told apart afterwards from the spectral centroid RainbowSTORM measures (`splitmeac.m`). `pipeline.run_multichannel_pipeline` + `srev.spectral` reproduce that:

* each channel is still labeled and blinked **independently** with its own `SrEvSimConfig` (labeling density, photophysics, seed) -- now also carrying a `dye` (`spectral.AF647` / `spectral.CF680`, or any `DyeSpectrum`);
* the channels' emitters are merged into one set and rendered into `spatial.tif` and `spectral.tif` (frame-aligned, same shape);
* a shared `SpectralConfig` describes the instrument: dispersion (nm/px), where a reference wavelength lands relative to the zeroth-order spot, the first-order PSF, the zeroth/first photon split, and the first-order background ratio;
* the merged ground truth (`resolved_localizations_gt.parquet`) carries `channel`, `dye` and `emission_centroid_nm` per blink, and `run_config.json` records the exact wavelength -> pixel-offset calibration to enter into RainbowSTORM.

The AF647 / CF680 presets are tuned so their centroids reproduce what RainbowSTORM measures for single-dye controls on the real instrument (`hmt_v1/Simulation/blink_stats`: ~670.5 nm and ~700 nm).

In [ ]:
dual_het = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=1000, mean_photons_per_frame=800.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="heterochromatin",
    dye=srev.spectral.AF647,
    seed=1,
)
dual_eu = srev.pipeline.SrEvSimConfig(
    labeling=srev.labeling.LabelingConfig(labeling_efficiency=0.3, n_dye_per_secondary=("poisson", 2.0)),
    photophysics=srev.photophysics.PhotophysicsConfig(n_frames=1000, mean_photons_per_frame=800.0),
    camera=srev.render.CameraConfig(pixel_size_nm=100.0),
    compartment_filter="euchromatin",
    dye=srev.spectral.CF680,
    seed=2,
)
# the instrument's spectral path -- shared by both channels. Defaults: 6 nm/px dispersion (legacy `SD=6`),
# 670 nm landing directly over the zeroth-order spot, 200 nm first-order PSF, 45/55 zeroth/first photon split.
spectral_config = srev.spectral.SpectralConfig()

result_dual = srev.pipeline.run_multichannel_pipeline(
    patch_df, [dual_het, dual_eu], spectral_config, OUTPUT_ROOT / "dual_color",
    source_path=CONFIG_PATH, input_meta=meta, seed=7,
)
for name, ch in result_dual["channels"].items():
    print(f"{name}: {len(ch['fluorophores'])} fluorophores, {len(ch['blink_events'])} blinks")
print(f"merged: {len(result_dual['fluorophores'])} fluorophores, {len(result_dual['blink_events'])} blinks, "
      f"frame {result_dual['frame_shape']} px")
print("spatial: ", result_dual["written"]["spatial_tiff"])
print("spectral:", result_dual["written"]["spectral_tiff"])

### The two stacks

Left: the zeroth-order movie, both dyes indistinguishable. Right: the first-order movie, every emitter smeared along +x by its spectrum -- a CF680 emitter's streak sits ~5 px (30 nm / 6 nm per px) redder than an AF647 one at the same position. The frame below picks an exposure where one emitter of each dye is on.

In [ ]:
spatial_stack = tifffile.imread(result_dual["written"]["spatial_tiff"])
spectral_stack = tifffile.imread(result_dual["written"]["spectral_tiff"])
gt_dual = result_dual["blink_events"].merge(
    result_dual["fluorophores"][["fluorophore_id", "x [nm]", "y [nm]", "dye"]], on="fluorophore_id")
px = dual_het.camera.pixel_size_nm

# an exposure with exactly one emitter of each dye on, at different rows
show_frame = 0
for f in range(spatial_stack.shape[0]):
    on = gt_dual[(gt_dual.frame_start <= f) & (gt_dual.frame_end >= f)]
    if len(on) == 2 and set(on.dye) == {"AF647", "CF680"} and abs(on["y [nm]"].diff().iloc[-1]) > 2.5 * px:
        show_frame = f
        break
on = gt_dual[(gt_dual.frame_start <= show_frame) & (gt_dual.frame_end >= show_frame)]

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes[0, 0].imshow(spatial_stack.max(axis=0), cmap="gray"); axes[0, 0].set_title("spatial.tif -- max projection")
axes[0, 1].imshow(spectral_stack.max(axis=0), cmap="gray"); axes[0, 1].set_title("spectral.tif -- max projection")
axes[1, 0].imshow(spatial_stack[show_frame], cmap="gray"); axes[1, 0].set_title(f"spatial frame {show_frame}")
axes[1, 1].imshow(spectral_stack[show_frame], cmap="gray"); axes[1, 1].set_title(f"spectral frame {show_frame}")
for _, row in on.iterrows():
    color = "r" if row.dye == "AF647" else "c"
    for ax in axes[1]:
        ax.plot(row["x [nm]"] / px, row["y [nm]"] / px, "o", mfc="none", mec=color, ms=14, label=row.dye)
axes[1, 1].legend(loc="upper right")
fig.tight_layout()
plt.show()

### Ground truth for scoring a RainbowSTORM run

`resolved_localizations_gt.parquet` is one row per blink with the TRUE position and the TRUE spectral centroid (`emission_centroid_nm`, the dye's centroid over `SpectralConfig.wavelength_window_nm` plus that molecule's own heterogeneity shift). Match RainbowSTORM's output to it on `(frame, x, y)` and compare its centroid column against this one; `run_config.json["spectral_calibration"]` has the wavelength -> pixel-offset relation (and a lookup table) the RainbowSTORM calibration should reproduce.

As a sanity check without leaving Python, `spectral.measure_spectral_centroids` does the crudest possible version of RainbowSTORM's centroid step (sum a few rows of the spectral frame over the detection window at the known position, subtract background, intensity-weighted mean wavelength). On blinks that are spectrally isolated in their frame it recovers the two dye populations; where two emitters' streaks overlap it doesn't -- which is real crosstalk, and the reason to simulate both dyes in one stack in the first place.

In [ ]:
gt_loaded = pd.read_parquet(result_dual["written"]["resolved_localizations_gt"])
display(gt_loaded[["blink_id", "frame_start", "x [nm]", "y [nm]", "z [nm]", "total_photons", "channel", "dye", "emission_centroid_nm"]].head())

measured_nm, measured_adu = srev.spectral.measure_spectral_centroids(
    result_dual["written"]["spectral_tiff"], gt_loaded["frame_start"], gt_loaded["x [nm]"], gt_loaded["y [nm]"],
    dual_het.camera, spectral_config, row_half_width_px=2, background="camera",
)
gt_loaded["measured_centroid_nm"] = measured_nm
bright = gt_loaded[np.isfinite(measured_nm) & (measured_adu > 250)]

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.arange(600, 780, 4)
for dye, color in (("AF647", "tab:red"), ("CF680", "tab:cyan")):
    sub = bright[bright.dye == dye]
    ax.hist(sub.measured_centroid_nm, bins=bins, alpha=0.6, color=color,
            label=f"{dye}: truth {sub.emission_centroid_nm.iloc[0]:.1f} nm, measured median {sub.measured_centroid_nm.median():.1f} nm (n={len(sub)})")
ax.axvline(685, color="k", ls="--", lw=1, label="685 nm split")
ax.set_xlabel("measured spectral centroid (nm)"); ax.set_ylabel("blinks"); ax.legend(fontsize=8)
plt.show()

### A calibration pair for RainbowSTORM

RainbowSTORM's spectral calibration is fitted from an image of point sources with lines at known wavelengths. `spectral.render_calibration_pair` renders exactly that -- pinholes in a spatial frame, one monochromatic line per wavelength per pinhole in the spectral frame -- with the same forward model as the data, so a calibration fitted from it applies to the stacks above.

In [ ]:
H, W = result_dual["frame_shape"]
pinhole_x = np.linspace(0.2, 0.5, 4) * W * px
pinhole_y = np.linspace(0.2, 0.8, 4) * H * px
cal_spatial, cal_spectral = srev.spectral.render_calibration_pair(
    pinhole_x, pinhole_y, (620, 660, 700, 740, 780), photons_per_line=3000.0, frame_shape=(H, W),
    camera_config=dual_het.camera, render_config=dual_het.render, spectral_config=spectral_config,
    rng=np.random.default_rng(0), output_dir=OUTPUT_ROOT / "dual_color" / "calibration",
)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(cal_spatial, cmap="gray"); axes[0].set_title("calibration_spatial.tif (pinholes)")
axes[1].imshow(cal_spectral, cmap="gray"); axes[1].set_title("calibration_spectral.tif (620 / 660 / 700 / 740 / 780 nm lines)")
fig.tight_layout()
plt.show()

## Next step (manual, outside this notebook)

Open each channel's `stack.tif` in ImageJ/Fiji, run ThunderSTORM's real detection + fitting, and nearest-neighbor-match its output against the corresponding `resolved_localizations_gt` table in `(x, y, frame)` (e.g. via `scipy.spatial.cKDTree`) to get a first detection-rate / localization-error number. This is the actual acceptance test for the whole pipeline's purpose -- everything above only prepares the inputs for it.

For the dual-color pair: run ThunderSTORM on `dual_color/spatial.tif`, calibrate RainbowSTORM from `dual_color/calibration/` (or enter `run_config.json["spectral_calibration"]` directly), extract spectral centroids from `dual_color/spectral.tif`, split the list on centroid (the legacy `splitmeac.m` step), and score both the localizations and the dye assignment against `dual_color/resolved_localizations_gt.parquet`'s `x/y/z` and `dye` / `emission_centroid_nm` columns.